# Task 05 — Map Qualcomm's Qwen adapter to the deployment stack

**Purpose.** Locate the real adapter classes and functions in the pinned
`qualcomm/ai-hub-models` source (tag `v0.59.0`), generate the two-part
boundary map, and compute the deployed graph contract from the source's
own rules.

**Environment.** Local, offline. Reads source files with `ast` only —
it never imports `qai_hub_models`. Needs `/Volumes/T9` mounted with the
pinned clone at `/Volumes/T9/qualcomm-edge-slm-lab/ai-hub-models`.

**Lesson:** [docs/tasks/05-qualcomm-qwen-adapter.html](../docs/tasks/05-qualcomm-qwen-adapter.html)


In [1]:
import ast
import json
import os
import subprocess
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = REPO / "results"
RESULTS.mkdir(exist_ok=True)

AIHUB = Path("/Volumes/T9/qualcomm-edge-slm-lab/ai-hub-models")
assert os.path.ismount("/Volumes/T9"), "Mount the T9 drive first."
assert AIHUB.is_dir(), f"clone missing: {AIHUB}"

tag = subprocess.run(
    ["git", "-C", str(AIHUB), "describe", "--tags"],
    capture_output=True, text=True, check=True,
).stdout.strip()
assert tag == "v0.59.0", f"clone is at {tag}, expected v0.59.0"
print("pinned clone OK:", AIHUB)
print("tag:", tag)

pinned clone OK: /Volumes/T9/qualcomm-edge-slm-lab/ai-hub-models
tag: v0.59.0


## Part 1 — The files

Five files in `qwen3_0_6b/` plus three shared modules carry the whole
story. Nothing here is hidden — it is plain Python on your disk.

**Inspect:** the line counts. The model-specific file is short because
the machinery lives in `_shared/`.

In [2]:
SRC = AIHUB / "src/qai_hub_models/models"

FILE_MAP = {
    "adapter (this model)":  SRC / "qwen3_0_6b/model.py",
    "export entry":          SRC / "qwen3_0_6b/export.py",
    "quantize entry":        SRC / "qwen3_0_6b/quantize.py",
    "calibration data":      SRC / "qwen3_0_6b/dataset.py",
    "README (Task 07 cmds)": SRC / "qwen3_0_6b/README.md",
    "shared Qwen3 base":     SRC / "_shared/qwen3/model.py",
    "shared Qwen3 patches":  SRC / "_shared/qwen3/model_adaptations.py",
    "shared LLM machinery":  SRC / "_shared/llm/model.py",
    "shared LLM common":     SRC / "_shared/llm/common.py",
    "shared LLM quantize":   SRC / "_shared/llm/quantize.py",
}

for role, path in FILE_MAP.items():
    assert path.is_file(), f"missing: {path}"
    n = len(path.read_text().splitlines())
    print(f"{role:24s} {n:5d} lines  {path.relative_to(AIHUB)}")

adapter (this model)       235 lines  src/qai_hub_models/models/qwen3_0_6b/model.py
export entry                69 lines  src/qai_hub_models/models/qwen3_0_6b/export.py
quantize entry              19 lines  src/qai_hub_models/models/qwen3_0_6b/quantize.py
calibration data           320 lines  src/qai_hub_models/models/qwen3_0_6b/dataset.py
README (Task 07 cmds)      134 lines  src/qai_hub_models/models/qwen3_0_6b/README.md
shared Qwen3 base          831 lines  src/qai_hub_models/models/_shared/qwen3/model.py
shared Qwen3 patches       379 lines  src/qai_hub_models/models/_shared/qwen3/model_adaptations.py
shared LLM machinery      3792 lines  src/qai_hub_models/models/_shared/llm/model.py
shared LLM common          183 lines  src/qai_hub_models/models/_shared/llm/common.py
shared LLM quantize        376 lines  src/qai_hub_models/models/_shared/llm/quantize.py


## Part 2 — Locate the required symbols

The locator parses each file with Python's `ast` module and walks the
syntax tree. Top-level classes, functions, and constants are recorded by
name. Methods are recorded as `Class.method`. Nothing is executed.

`REQUIRED_SYMBOLS` is the list the completion gate checks.

In [3]:
def locate_symbols(path):
    """Map every top-level class, function, constant, and class method
    in one Python file to its line number."""
    tree = ast.parse(path.read_text())
    found = {}
    for node in tree.body:
        if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
            found[node.name] = node.lineno
            if isinstance(node, ast.ClassDef):
                for sub in node.body:
                    if isinstance(sub, ast.FunctionDef):
                        found[f"{node.name}.{sub.name}"] = sub.lineno
        elif isinstance(node, ast.Assign):
            for t in node.targets:
                if isinstance(t, ast.Name):
                    found[t.id] = node.lineno
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name):
            found[node.target.id] = node.lineno
    return found


REQUIRED_SYMBOLS = {
    "qwen3_0_6b/model.py": [
        "NUM_LAYERS", "NUM_SPLITS", "SPINQUANT_CONFIG", "SPLIT_MODEL_NAME",
        "Qwen3_0_6B_PreSplit",
        "Qwen3_0_6B_QuantizablePreSplit",
        "Qwen3_0_6B_QuantizablePreSplit.get_calibration_data",
        "Qwen3_0_6B_QuantizablePreSplit.get_weight_optimization_data",
        "Qwen3_0_6B_Part1_Of_2",
        "Qwen3_0_6B_Part2_Of_2",
        "Qwen3_0_6B_Collection",
    ],
    "qwen3_0_6b/export.py": [
        "DEFAULT_EXPORT_DEVICE", "build_parser", "main",
    ],
    "_shared/qwen3/model.py": [
        "Qwen3Base", "Qwen3Base.monkey_patch",
        "Qwen3PositionProcessor", "Qwen3PositionProcessor.forward",
        "Qwen3PreSplitBase", "Qwen3QuantizablePreSplitBase",
        "Qwen3PartBase", "Qwen3PreSplitCollectionBase",
    ],
    "_shared/qwen3/model_adaptations.py": [
        "SHAQwen3Attention", "SHAQwen3Attention.prepare_conv",
        "QcQwen3_apply_rotary_pos_emb", "QCQwen3MLP", "QCQwen3ForCausalLM",
    ],
    "_shared/llm/model.py": [
        "LLMBase", "LLMBase.get_input_spec", "LLMBase._get_input_spec",
        "SplitForwardMixin",
        "DEFAULT_SEQUENCE_LENGTH", "DEFAULT_CONTEXT_LENGTH",
        "DEFAULT_EXPORT_SEQUENCE_LENGTHS",
    ],
    "_shared/llm/common.py": ["LLMIOType"],
    "_shared/llm/quantize.py": ["llm_quantize"],
}

source_map = {}
missing = []
for rel, names in REQUIRED_SYMBOLS.items():
    found = locate_symbols(SRC / rel)
    for name in names:
        if name in found:
            source_map[name] = {"file": f"src/qai_hub_models/models/{rel}",
                                "line": found[name]}
        else:
            missing.append(f"{rel}: {name}")

assert not missing, "not located: " + ", ".join(missing)

n_required = sum(len(v) for v in REQUIRED_SYMBOLS.values())
print(f"located {len(source_map)} of {n_required} required symbols\n")
for name, loc in source_map.items():
    print(f"{name:55s} {loc['file'].split('models/', 1)[1]}:{loc['line']}")

(RESULTS / "05_source_map.json").write_text(json.dumps(source_map, indent=2))
print("\nsaved results/05_source_map.json")

located 36 of 36 required symbols

NUM_LAYERS                                              models/qwen3_0_6b/model.py:67
NUM_SPLITS                                              models/qwen3_0_6b/model.py:70
SPINQUANT_CONFIG                                        models/qwen3_0_6b/model.py:93
SPLIT_MODEL_NAME                                        models/qwen3_0_6b/model.py:96
Qwen3_0_6B_PreSplit                                     models/qwen3_0_6b/model.py:99
Qwen3_0_6B_QuantizablePreSplit                          models/qwen3_0_6b/model.py:121
Qwen3_0_6B_QuantizablePreSplit.get_calibration_data     models/qwen3_0_6b/model.py:144
Qwen3_0_6B_QuantizablePreSplit.get_weight_optimization_data models/qwen3_0_6b/model.py:161
Qwen3_0_6B_Part1_Of_2                                   models/qwen3_0_6b/model.py:191
Qwen3_0_6B_Part2_Of_2                                   models/qwen3_0_6b/model.py:197
Qwen3_0_6B_Collection                                   models/qwen3_0_6b/model.py:225
DEFAULT_E

**Inspect:** pick two rows and open them in your editor, for example
`Qwen3Base.monkey_patch` and `LLMBase._get_input_spec`. Read about 30
lines at each. The lesson's sections 5.3 and 5.4 walk the same code.

## Part 3 — The architecture constants

Read the constants straight out of `qwen3_0_6b/model.py` (as literal
values, still without importing) and check them against the real config
you measured in Task 04.

In [4]:
def literal_constants(path):
    """Top-level NAME = <literal> assignments in one file."""
    consts = {}
    for node in ast.parse(path.read_text()).body:
        if (isinstance(node, ast.Assign) and len(node.targets) == 1
                and isinstance(node.targets[0], ast.Name)):
            try:
                consts[node.targets[0].id] = ast.literal_eval(node.value)
            except ValueError:
                pass  # not a literal (e.g. Precision.w4a16) — skip
    return consts


consts = literal_constants(SRC / "qwen3_0_6b/model.py")

# The same numbers your Task 04 notebook pulled from AutoConfig:
assert consts["NUM_LAYERS"] == 28
assert consts["HIDDEN_SIZE"] == 1024
assert consts["NUM_ATTN_HEADS"] == 16
assert consts["NUM_KEY_VALUE_HEADS"] == 8
assert consts["HEAD_DIM"] == 128
assert consts["NUM_SPLITS"] == 2
assert consts["HF_REPO_NAME"] == "Qwen/Qwen3-0.6B"
assert consts["SPINQUANT_CONFIG"] == {
    "enable_r1": False, "enable_r2": True, "enable_r3": True}

for k in ["NUM_LAYERS", "NUM_SPLITS", "NUM_LAYERS_PER_SPLIT", "HIDDEN_SIZE",
          "NUM_ATTN_HEADS", "NUM_KEY_VALUE_HEADS", "HEAD_DIM",
          "HF_REPO_NAME", "SPINQUANT_CONFIG", "SPLIT_MODEL_NAME"]:
    print(f"{k:22s} = {consts[k]}")
print("\nconstants match the config you measured in Task 04")

NUM_LAYERS             = 28
NUM_SPLITS             = 2
NUM_LAYERS_PER_SPLIT   = 28
HIDDEN_SIZE            = 1024
NUM_ATTN_HEADS         = 16
NUM_KEY_VALUE_HEADS    = 8
HEAD_DIM               = 128
HF_REPO_NAME           = Qwen/Qwen3-0.6B
SPINQUANT_CONFIG       = {'enable_r1': False, 'enable_r2': True, 'enable_r3': True}
SPLIT_MODEL_NAME       = Qwen3_0_6B

constants match the config you measured in Task 04


## Part 4 — The two-part boundary map

`Qwen3_0_6B_Collection.parts` registers the deployed pieces. The part
descriptions below are read from the class docstrings — generated, not
typed in.

The second section maps each deployment stage to the file and symbols
that serve it. This is the task's main output.

In [5]:
tree = ast.parse((SRC / "qwen3_0_6b/model.py").read_text())
parts_reg, part_ids, part_docs = {}, {}, {}
for node in tree.body:
    if not isinstance(node, ast.ClassDef):
        continue
    doc = ast.get_docstring(node)
    if doc:
        part_docs[node.name] = doc.splitlines()[0]
    for sub in node.body:
        if (isinstance(sub, ast.Assign) and len(sub.targets) == 1
                and isinstance(sub.targets[0], ast.Name)):
            if sub.targets[0].id == "parts":
                parts_reg = {k.value: v.id for k, v in
                             zip(sub.value.keys, sub.value.values)}
            elif sub.targets[0].id == "part_id":
                part_ids[node.name] = ast.literal_eval(sub.value)

assert list(parts_reg) == ["part1_of_2", "part2_of_2"]
assert [part_ids[c] for c in parts_reg.values()] == [1, 2]

boundary_map = {
    "model": consts["HF_REPO_NAME"],
    "pinned_tag": tag,
    "num_splits": consts["NUM_SPLITS"],
    "split_file_basename": consts["SPLIT_MODEL_NAME"],
    "parts": {
        key: {
            "class": cls,
            "part_id": part_ids[cls],
            "contains": part_docs[cls],
            "file": source_map[cls]["file"],
            "line": source_map[cls]["line"],
        }
        for key, cls in parts_reg.items()
    },
    "stack": {
        "1_adapt": {
            "what": "fill in 0.6B constants; register the two parts",
            "file": source_map["Qwen3_0_6B_Collection"]["file"],
            "symbols": ["Qwen3_0_6B_PreSplit", "Qwen3_0_6B_Collection"],
        },
        "2_patch": {
            "what": "swap HF modules for NPU-friendly forms (SHA, RoPE bypass)",
            "file": source_map["Qwen3Base.monkey_patch"]["file"],
            "symbols": ["Qwen3Base.monkey_patch", "SHAQwen3Attention",
                        "QcQwen3_apply_rotary_pos_emb"],
        },
        "3_contract": {
            "what": "declare the static graph inputs/outputs",
            "file": source_map["LLMBase._get_input_spec"]["file"],
            "symbols": ["LLMBase._get_input_spec", "LLMIOType"],
        },
        "4_quantize": {
            "what": "SpinQuant R2+R3 -> AdaScale (WikiText) -> "
                    "calibration (InterleavedGeneratedWikitext)",
            "file": "src/qai_hub_models/models/qwen3_0_6b/quantize.py",
            "symbols": ["llm_quantize", "SPINQUANT_CONFIG",
                        "Qwen3_0_6B_QuantizablePreSplit.get_weight_optimization_data",
                        "Qwen3_0_6B_QuantizablePreSplit.get_calibration_data"],
            "spinquant_config": consts["SPINQUANT_CONFIG"],
        },
        "5_export": {
            "what": "compile for a device on AI Hub Workbench",
            "file": "src/qai_hub_models/models/qwen3_0_6b/export.py",
            "symbols": ["build_parser", "main"],
        },
    },
}

for s in boundary_map["stack"].values():
    for sym in s["symbols"]:
        assert sym in source_map, f"stack references unlocated symbol {sym}"

(RESULTS / "05_boundary_map.json").write_text(json.dumps(boundary_map, indent=2))
print(json.dumps(boundary_map["parts"], indent=2))
print("\nstack stages:", ", ".join(boundary_map["stack"]))
print("saved results/05_boundary_map.json")

{
  "part1_of_2": {
    "class": "Qwen3_0_6B_Part1_Of_2",
    "part_id": 1,
    "contains": "Part 1: Embedding.",
    "file": "src/qai_hub_models/models/qwen3_0_6b/model.py",
    "line": 191
  },
  "part2_of_2": {
    "class": "Qwen3_0_6B_Part2_Of_2",
    "part_id": 2,
    "contains": "Part 2: Transformer layers + LM head.",
    "file": "src/qai_hub_models/models/qwen3_0_6b/model.py",
    "line": 197
  }
}

stack stages: 1_adapt, 2_patch, 3_contract, 4_quantize, 5_export
saved results/05_boundary_map.json


**Inspect:** the `contains` lines. They came from the docstrings of
`Qwen3_0_6B_Part1_Of_2` and `Qwen3_0_6B_Part2_Of_2`. Part 1 is only the
embedding. Part 2 is the 28 layers plus the LM head.

## Part 5 — The deployed graph contract

`LLMBase._get_input_spec` (you located it above — read it now) builds
the contract from six numbers. The cell below applies the same rules
with the constants read in Part 3.

Two graphs ship: `DEFAULT_EXPORT_SEQUENCE_LENGTHS = [128, 1]` — prefill
processes 128 tokens per pass, decode processes 1.

In [9]:
# PARAMETERS — you can change these
SEQUENCE_LENGTH = 1   # run once with 128 (prefill), then rerun with 1 (decode)
CONTEXT_LENGTH = 4096   # DEFAULT_CONTEXT_LENGTH; leave as is

In [10]:
L, C = SEQUENCE_LENGTH, CONTEXT_LENGTH
assert L < C, "the source asserts sequence_length < context_length"

N = consts["NUM_LAYERS"]
H_KV = consts["NUM_KEY_VALUE_HEADS"]
D = consts["HEAD_DIM"]
EMBED = D // 2  # cos/sin each cover half of head_dim

# Same rules as LLMBase._get_input_spec (llm_io_type=genie_input_ids):
inputs = {
    "input_ids":        ((1, L), "int32"),
    "attention_mask":   ((1, 1, L, C), "float32"),
    "position_ids_cos": ((1, 1, L, EMBED), "float32"),
    "position_ids_sin": ((1, 1, L, EMBED), "float32"),
}
for i in range(N):
    # key stored transposed: head_dim before the cache slots
    inputs[f"past_key_{i}_in"] = ((H_KV, 1, D, C - L), "float32")
    inputs[f"past_value_{i}_in"] = ((H_KV, 1, C - L, D), "float32")

outputs = ["logits"] + [
    f"past_{t}_{i}_out" for i in range(N) for t in ("key", "value")]

assert len(inputs) == 4 + 2 * N == 60
assert len(outputs) == 1 + 2 * N == 57
assert inputs[f"past_key_0_in"][0] == (8, 1, 128, C - L)
assert inputs[f"past_value_0_in"][0] == (8, 1, C - L, 128)

graph = "prefill" if L > 1 else "decode"
print(f"{graph} graph (sequence_length={L}, context_length={C})")
print(f"{len(inputs)} inputs, {len(outputs)} outputs")
print(f"cache slots per input buffer: {C - L}\n")
for name in ["input_ids", "attention_mask", "position_ids_cos",
             "position_ids_sin", "past_key_0_in", "past_value_0_in"]:
    shape, dtype = inputs[name]
    print(f"{name:18s} {str(shape):22s} {dtype}")
print(f"... and {2 * N - 2} more cache inputs")

# merge-save, keyed by sequence length (both runs survive)
out_path = RESULTS / "05_graph_contract.json"
saved = json.loads(out_path.read_text()) if out_path.exists() else {}
saved[str(L)] = {
    "graph": graph,
    "context_length": C,
    "num_inputs": len(inputs),
    "num_outputs": len(outputs),
    "inputs_head": {k: {"shape": list(v[0]), "dtype": v[1]}
                    for k, v in list(inputs.items())[:6]},
}
out_path.write_text(json.dumps(saved, indent=2))
print(f"\nsaved under key '{L}' -> results/05_graph_contract.json")

decode graph (sequence_length=1, context_length=4096)
60 inputs, 57 outputs
cache slots per input buffer: 4095

input_ids          (1, 1)                 int32
attention_mask     (1, 1, 1, 4096)        float32
position_ids_cos   (1, 1, 1, 64)          float32
position_ids_sin   (1, 1, 1, 64)          float32
past_key_0_in      (8, 1, 128, 4095)      float32
past_value_0_in    (8, 1, 4095, 128)      float32
... and 54 more cache inputs

saved under key '1' -> results/05_graph_contract.json


**Inspect:** the cache-slot count. At 128 the buffers hold 3968
slots; at 1 they hold 4095. `context_length - sequence_length` — the
current chunk's K/V leaves the graph as output instead.

In [11]:
# Your Task 04 contract vs the deployed one, side by side.
t04 = json.loads((REPO / "results/04_contracts.json").read_text())
t04_decode = t04["qwen3_0_6b_from_config"]["decode"]

print("decode step, same model, two designs\n")
print(f"{'':22s}{'yours (Task 04)':28s}deployed (v0.59.0)")
print(f"{'inputs':22s}{t04_decode['num_inputs']:<28d}60")
print(f"{'outputs':22s}{t04_decode['num_outputs']:<28d}57")
print(f"{'positions':22s}{'implicit (cache length)':28s}cos/sin, host-computed")
print(f"{'mask':22s}{'none (dense causal)':28s}additive, clip(-50, 0)")
key04 = t04_decode["past_key_i / past_value_i (x28 layers)"]
print(f"{'past key layout':22s}{str(key04):28s}(8, 1, 128, 4095)")
print(f"{'cache growth':22s}{'output is 1 longer':28s}output = new slice only")
print("\nSame job. The deployed layout is frozen, transposed, and padded")
print("to context_length — your Part 6 design, at real scale.")

decode step, same model, two designs

                      yours (Task 04)             deployed (v0.59.0)
inputs                57                          60
outputs               57                          57
positions             implicit (cache length)     cos/sin, host-computed
mask                  none (dense causal)         additive, clip(-50, 0)
past key layout       [1, 8, 24, 128]             (8, 1, 128, 4095)
cache growth          output is 1 longer          output = new slice only

Same job. The deployed layout is frozen, transposed, and padded
to context_length — your Part 6 design, at real scale.


## Rerun with the decode graph

Go back to the **PARAMETERS** cell, set `SEQUENCE_LENGTH = 1`, and run
from there through the summary below. The contract file then holds both
graphs.

In [12]:
# FINAL SUMMARY — reports the completion-gate checks
contract = json.loads((RESULTS / "05_graph_contract.json").read_text())
bmap = json.loads((RESULTS / "05_boundary_map.json").read_text())

CHECKS = {
    "clone_pinned_v0_59_0": tag == "v0.59.0",
    "all_symbols_located": len(source_map) == n_required,
    "constants_match_task04": (
        consts["NUM_LAYERS"] == 28 and consts["NUM_KEY_VALUE_HEADS"] == 8
        and consts["NUM_ATTN_HEADS"] == 16 and consts["HEAD_DIM"] == 128
        and consts["HIDDEN_SIZE"] == 1024 and consts["NUM_SPLITS"] == 2),
    "boundary_map_two_parts": (
        set(bmap["parts"]) == {"part1_of_2", "part2_of_2"}
        and {p["part_id"] for p in bmap["parts"].values()} == {1, 2}),
    "boundary_map_stack_stages": len(bmap["stack"]) == 5,
    "contract_both_graphs": {"128", "1"} <= set(contract),
    "decode_contract_60_in_57_out": (
        "1" in contract and contract["1"]["num_inputs"] == 60
        and contract["1"]["num_outputs"] == 57),
}

summary = {
    "task": 5,
    "checks": CHECKS,
    "symbols_located": len(source_map),
    "artifacts": ["results/05_source_map.json",
                  "results/05_boundary_map.json",
                  "results/05_graph_contract.json"],
}
(RESULTS / "05_summary.json").write_text(json.dumps(summary, indent=2))

for name, ok in CHECKS.items():
    print(f"{'PASS' if ok else 'FAIL':4s} {name}")
print("\nsaved results/05_summary.json")
if not all(CHECKS.values()):
    print("Some checks are False. If contract_both_graphs failed, rerun "
          "from the PARAMETERS cell with SEQUENCE_LENGTH = 1.")

PASS clone_pinned_v0_59_0
PASS all_symbols_located
PASS constants_match_task04
PASS boundary_map_two_parts
PASS boundary_map_stack_stages
PASS contract_both_graphs
PASS decode_contract_60_in_57_out

saved results/05_summary.json
